In [1]:
import pandas as pd

In [30]:
# read in ssm data

ssm = pd.read_excel('./ssm_model_obs_pair.xlsx', sheet_name='Sheet1')

In [3]:
print(ssm)

       Station          Model_Time  Nodes   ID Type  Depth_m Masked_Area?  \
0       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      1.0           No   
1       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      1.5           No   
2       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      2.0           No   
3       ADM001 2014-03-13 12:00:00   6231  MMU  Lab      1.5           No   
4       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      2.5           No   
...        ...                 ...    ...  ...  ...      ...          ...   
101239  FID001 2014-11-18 12:00:00   5830  MMU  CTD      8.0           No   
101240  FID001 2014-11-18 12:00:00   5830  MMU  CTD      8.5           No   
101241  FID001 2014-11-18 12:00:00   5830  MMU  CTD      9.0           No   
101242  FID001 2014-11-18 12:00:00   5830  MMU  CTD     11.5           No   
101243  FID001 2014-11-18 12:00:00   5830  MMU  CTD     12.0           No   

        Temp_C  Salinity_psu  DO_mgL  ...  Model_DO_mgL  Model_NO23N_mgL  \

In [13]:
ssm.columns

Index(['name', 'time', 'Nodes', 'ID', 'Type', 'z', 'Masked_Area?', 'Temp_C',
       'Salinity_psu', 'DO_mgL', 'Chla_ugL', 'NO23N_mgL', 'NH4N_mgL',
       'PAR_Em2day', 'Layer', 'CT', 'SA', 'DO', 'NO3', 'NH4',
       'Model_PAR_Em2day', 'Chl', 'Layer_Depth_m', 'Embayment', 'Layer_Cat',
       'Hydro_T', 'WQM_T'],
      dtype='object')

In [31]:
# read in 2014 lo_ssc data
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc.pkl")

In [26]:
print(data["obs"])

         cid         lon        lat                time          z         SA  \
0        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.148984   
1        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149086   
2        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149387   
3        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149889   
4        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -5.255167  31.145790   
...      ...         ...        ...                 ...        ...        ...   
5013  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -24.400000        NaN   
5014  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -33.900000        NaN   
5015  3351.0 -122.428001  47.744000 2014-12-15 17:42:00 -14.900000        NaN   
5016  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -0.620000        NaN   
5017  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -1.600000        NaN   

            CT          DO 

In [16]:
data["obs"].columns

Index(['cid', 'lon', 'lat', 'time', 'z', 'SA', 'CT', 'DO', 'NO3', 'Chl',
       'name', 'cruise', 'source', 'NH4', 'PO4 (uM)', 'SiO4 (uM)', 'NO2 (uM)',
       'TA', 'DIC'],
      dtype='object')

In [32]:
# rename ssm columns to match the other dataframes

ssm = ssm.rename(columns={
    'Station': 'name',
    'Depth_m': 'z',
    'Model_Time': 'time',
    'Model_Temp_C': 'CT',
    'Model_Salinity_psu': 'SA',
    'Model_DO_mgL': 'DO',
    'Model_Chla_ugL': 'Chl',
    'Model_NO23N_mgL': 'NO3',
    'Model_NH4N_mgL': 'NH4'}) 

# convert DO from mg/L to umol/L 
ssm["DO"] = ssm["DO"] * 1000 / 31.998 

# convert NH4 from mg/L to umol/L 
ssm["NH4"] = ssm["NH4"] * 1000 / 18.039 

# convert NO3 from mg/L to umol/L 
ssm["NO3"] = ssm["NO3"] * 1000 / 62.004 

# make depth negative 
ssm["z"] = -ssm["z"] 

# ensure datetime is in datetime format 
ssm["time"] = pd.to_datetime(ssm["time"]) 

# standardize depth to float 
ssm["z"] = ssm["z"].astype(float)

In [33]:
# copies
obs = data['obs'].copy()
ssm_df = ssm.copy()

# formats
obs['time'] = pd.to_datetime(obs['time'])
ssm_df['time'] = pd.to_datetime(ssm_df['time'])

obs['z'] = obs['z'].round(2)
ssm_df['z'] = ssm_df['z'].round(2)

# marker indicating a successful obs match
obs_match = obs[['cid','name', 'z', 'time']].copy()
obs_match['matched_obs'] = True

# sort (required)
obs_match = obs_match.sort_values('time')
ssm_df = ssm_df.sort_values('time')

# asof merge with tolerance
ssm_matched = pd.merge_asof(
    ssm_df,
    obs_match,
    on='time',
    by=['name','z'],
    direction='nearest',
    tolerance=pd.Timedelta('3hr')
)

ssm_matched = ssm_matched[
    ssm_matched['matched_obs'].notna()
].drop(columns='matched_obs')

In [34]:
# keep only standardized columns
keep_cols = [
    'cid','name', 'time', 'z',
    'CT', 'SA', 'DO', 'NO3', 'NH4', 'Chl'
]

ssm_matched = ssm_matched[keep_cols]

In [35]:
print(ssm_matched)
ssm_matched.columns

           cid    name                time      z         CT         SA  \
3111    1843.0  KSBP01 2014-01-21 09:00:00 -197.0   9.047412  29.966526   
3182    1849.0  CK200P 2014-01-21 09:00:00  -34.6   9.716570  29.766726   
3223    1849.0  CK200P 2014-01-21 09:00:00  -24.7   9.770035  29.604378   
3227    1843.0  KSBP01 2014-01-21 09:00:00  -34.5   9.525203  29.800238   
3349    1843.0  KSBP01 2014-01-21 09:00:00  -99.0   9.281917  29.925127   
...        ...     ...                 ...    ...        ...        ...   
101214  1229.0  NSEX01 2014-12-16 13:00:00  -54.7  11.023428  30.078165   
101215  1229.0  NSEX01 2014-12-16 13:00:00  -34.8  10.964799  30.014685   
101216  1229.0  NSEX01 2014-12-16 13:00:00  -24.9  10.958919  29.981766   
101217  1229.0  NSEX01 2014-12-16 13:00:00  -14.9  10.921903  29.850718   
101219  1229.0  NSEX01 2014-12-16 13:00:00   -1.8   9.138308  25.005932   

                DO       NO3       NH4       Chl  
3111    244.366377  6.855410  0.701883  0.141081

Index(['cid', 'name', 'time', 'z', 'CT', 'SA', 'DO', 'NO3', 'NH4', 'Chl'], dtype='object')

In [36]:
matched_cids = set(ssm_matched['cid'])

mask = data['obs']['cid'].isin(matched_cids)

for k in ['obs', 'cas7_t1_x11ab', 'ssc']:
    data[k] = data[k].loc[mask].reset_index(drop=True)

data['ssm'] = ssm_matched.reset_index(drop=True)

In [37]:
for k in ['obs', 'cas7_t1_x11ab', 'ssc']:
    print(k, len(data[k]), data[k].index[:5])

obs 1078 RangeIndex(start=0, stop=5, step=1)
cas7_t1_x11ab 1078 RangeIndex(start=0, stop=5, step=1)
ssc 1078 RangeIndex(start=0, stop=5, step=1)


In [38]:
print(data['ssm'])

         cid    name                time      z         CT         SA  \
0     1843.0  KSBP01 2014-01-21 09:00:00 -197.0   9.047412  29.966526   
1     1849.0  CK200P 2014-01-21 09:00:00  -34.6   9.716570  29.766726   
2     1849.0  CK200P 2014-01-21 09:00:00  -24.7   9.770035  29.604378   
3     1843.0  KSBP01 2014-01-21 09:00:00  -34.5   9.525203  29.800238   
4     1843.0  KSBP01 2014-01-21 09:00:00  -99.0   9.281917  29.925127   
...      ...     ...                 ...    ...        ...        ...   
1369  1229.0  NSEX01 2014-12-16 13:00:00  -54.7  11.023428  30.078165   
1370  1229.0  NSEX01 2014-12-16 13:00:00  -34.8  10.964799  30.014685   
1371  1229.0  NSEX01 2014-12-16 13:00:00  -24.9  10.958919  29.981766   
1372  1229.0  NSEX01 2014-12-16 13:00:00  -14.9  10.921903  29.850718   
1373  1229.0  NSEX01 2014-12-16 13:00:00   -1.8   9.138308  25.005932   

              DO       NO3       NH4       Chl  
0     244.366377  6.855410  0.701883  0.141081  
1     241.312655  7.17221

In [39]:
# save the combined data
pd.to_pickle(data, 'combined_bottle_2014_cas7_t1_x11ab_ssc_ssm.pkl')